<a href="https://colab.research.google.com/github/francji1/01RAD/blob/main/HW/01RAD_Ex02_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01RAD - Homework 02: Regression to the mean (Galton's heights)

In 1886 Francis Galton compared the heights of 1,078 fathers and their adult sons and noticed that sons of tall fathers tend to be shorter than their fathers, and sons of short fathers taller. He called the phenomenon *regression towards mediocrity*; the name stuck to the whole method (Lecture 1, history). The data are the `fsdata.csv` file from Exercise 01 (heights in inches).

Model of Lecture 1: $Y_i = \beta_0 + \beta_1 x_i + e_i$ with $x_i$ the father's height and $Y_i$ the son's height, independent errors $e_i \sim (0, \sigma^2)$. Tools of Lecture 2 (slides pp. 14–32 of the lecture PDF): standard errors $\widehat{\sigma}(\widehat{\beta}_j) = s_n \sqrt{\delta_j}$, **confidence intervals** (Czech: intervaly spolehlivosti) $\widehat{\beta}_j \pm t_{1-\alpha/2}(n-2)\,\widehat{\sigma}(\widehat{\beta}_j)$ and the **tests** (testy hypotéz) with statistics $T_j = (\widehat{\beta}_j - b)/\widehat{\sigma}(\widehat{\beta}_j)$ compared with $t_{1-\alpha/2}(n-2)$.

## Submission
Work through the tasks in this notebook (code plus short written answers in markdown cells; Python or R, as you prefer). Submit through GitHub **before the next exercise**: fork `francji1/01RAD`, save the notebook into the `HW/` folder as `01RAD_Ex02_HW_<Surname>.ipynb`, with your surname in place of `<Surname>` and without the brackets, for example `01RAD_Ex02_HW_Novak.ipynb` (team: surnames joined by an underscore), and open a pull request to `main`. The pull request should add your one file only; do not modify or delete other files. A selected student solution, and a reference solution if needed, will be published in `HW/` a week later.

Useful calls: `stats.t.ppf(1 - alpha/2, df)` for the $t$ quantile, `stats.t.sf(abs(T), df)` and `stats.t.cdf(T, df)` for tail probabilities, `model.conf_int(alpha)`, `model.t_test('father = 1')`, `model.predict(pd.DataFrame({'father': [...]}))`, `np.corrcoef`, `df.sample(n, random_state=...)`.

## Data

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', palette='deep')

url_fsdata = 'https://raw.githubusercontent.com/francji1/01RAD/main/data/fsdata.csv'
fsdata = pd.read_csv(url_fsdata, index_col=0)
print(f'Rows: {fsdata.shape[0]}, columns: {fsdata.shape[1]}')
fsdata.describe().T

Rows: 1078, columns: 2


,count,mean,std,min,25%,50%,75%,max
father,1078.0,67.687097,2.744868,59.00800,65.787735,67.76660,69.60298,75.43393
son,1078.0,68.684070,2.814702,58.50708,66.931232,68.61582,70.46597,78.36479


## Task 01 - Fit and recap
Fit the regression of `son` on `father`. Compute $\widehat{\beta}_0$, $\widehat{\beta}_1$, $s_n$ and both standard errors by hand (formulas of Lectures 1 and 2) and check them against `statsmodels` (R: `summary(lm(son ~ father))`). Interpret $\widehat{\beta}_1$ in one sentence.

In [21]:
x = fsdata['father'].to_numpy(dtype=float)
y = fsdata['son'].to_numpy(dtype=float)
n = len(x)
x_bar, y_bar = x.mean(), y.mean()
S_xx = np.sum((x - x_bar) ** 2)
beta1_hat = np.sum((x - x_bar) * y) / S_xx
beta0_hat = y_bar - beta1_hat * x_bar
resid = y - (beta0_hat + beta1_hat * x)
SSE = np.sum(resid ** 2)
s = np.sqrt(SSE / (n - 2))
se_beta0 = s * np.sqrt(1 / n + x_bar ** 2 / S_xx)
se_beta1 = s * np.sqrt(1 / S_xx)

model = smf.ols('son ~ father', data=fsdata).fit()

comparison = pd.DataFrame({
    "Metric": ['beta0_hat', 'beta1_hat', 's', 'SE(beta0_hat)', 'SE(beta1_hat)'],
    "Manual": [beta0_hat, beta1_hat, s, se_beta0, se_beta1],
    "Statsmodels": [model.params['Intercept'], model.params['father'], np.sqrt(model.scale), model.bse['Intercept'], model.bse['father']]
})
print(comparison)


          Metric     Manual  Statsmodels
0      beta0_hat  33.886604    33.886604
1      beta1_hat   0.514093     0.514093
2              s   2.436556     2.436556
3  SE(beta0_hat)   1.832354     1.832354
4  SE(beta1_hat)   0.027049     0.027049


## Task 02 - Confidence intervals
Compute the 95 % and the 99 % confidence intervals for $\beta_0$ and $\beta_1$ by hand, using the $t$ quantile, and compare with `conf_int` (R: `confint`). Why is the 99 % interval wider? Compare the widths of the intervals for $\beta_0$ and $\beta_1$ in absolute terms and relative to the estimates: which parameter is estimated more precisely? Relate your answer to the remark in Lecture 2.

 kvantil t-rozdělení pro 99 % je větší než pro 95 %. Standardní chybu tedy násobíme větším číslem.V absolutních hodnotách interval pro směrnici $\beta_1$ (šířka $\approx 0.106$) je mnohem užší než interval pro absolutní člen $\beta_0$ (šířka $\approx 7.191$). Proto, pokud se díváme pouze na absolutní čísla, je parametr $\beta_1$ odhadnut s mnohem větší přesností. Relativní odhad: pokud porovnáme šířku intervalu s velikostí samotného koeficientu, relativní přesnost se ukázuje téměř totožná. Šířka intervalu pro $\beta_0$ tvoří přibližně $21.2\ \%$ jeho hodnoty ($33.88$), zatímco šířka intervalu pro $\beta_1$ tvoří přibližně $20.6\ \%$ jeho hodnoty ($0.51$).

In [22]:
alpha1 = 0.05
alpha0 = 0.01
t_95 = stats.t.ppf(0.975, df=n - 2)
t_99 = stats.t.ppf(0.995, df=n - 2)

ci_95_manual = pd.DataFrame({
    'lower': [beta0_hat - t_95 * se_beta0, beta1_hat - t_95 * se_beta1],
    'upper': [beta0_hat + t_95 * se_beta0, beta1_hat + t_95 * se_beta1]
}, index=['Intercept', 'father'])

ci_99_manual = pd.DataFrame({
    'lower': [beta0_hat - t_99 * se_beta0, beta1_hat - t_99 * se_beta1],
    'upper': [beta0_hat + t_99 * se_beta0, beta1_hat + t_99 * se_beta1]
}, index=['Intercept', 'father'])

print("--- 95% Confidence Intervals ---")
print("Manual:")
print(ci_95_manual)
print("\nStatsmodels:")
print(model.conf_int(alpha=0.05))
print("\n--- 99% Confidence Intervals ---")
print("Manual:")
print(ci_99_manual)
print("\nStatsmodels:")
print(model.conf_int(alpha=0.01))


--- 95% Confidence Intervals ---
Manual:
               lower      upper
Intercept  30.291213  37.481996
father      0.461019   0.567167

Statsmodels:
                   0          1
Intercept  30.291213  37.481996
father      0.461019   0.567167

--- 99% Confidence Intervals ---
Manual:
               lower      upper
Intercept  29.158387  38.614822
father      0.444296   0.583890

Statsmodels:
                   0          1
Intercept  29.158387  38.614822
father      0.444296   0.583890


## Task 03 - Is the father's height informative?
Test $H_0: \beta_1 = 0$ against $H_1: \beta_1 \neq 0$ at $\alpha = 0.05$: report $T_1$, the critical value $t_{0.975}(n-2)$, the $p$-value and the decision. What does the rejection say about the relationship, and what does it *not* say (think of the lecture's warning about linearity)?

Mezi výškou otce a výškou syna existuje lineární vztah. Zamítnutí hypotézy $\beta_1 = 0$ říká pouze o přítomnosti lineárního trendu. Neříká to však, že vztah je dokonale lineární.

In [23]:
T1 = beta1_hat / se_beta1
t_crit = stats.t.ppf(0.975, df=n - 2)
p_value = 2 * stats.t.sf(abs(T1), df=n - 2)

print(f"T1 = {T1:.4f}")
print(f"Critical value t_0.975({n-2}) = {t_crit:.4f}")
print(f"p-value = {p_value:.2e}")

if abs(T1) > t_crit:
    print("Decision: Reject H0")
else:
    print("Decision: Do not reject H0")


T1 = 19.0062
Critical value t_0.975(1076) = 1.9622
p-value = 1.12e-69
Decision: Reject H0


## Task 04 - Regression to the mean
If sons were on average exactly as tall as their fathers plus a constant, the slope would be $\beta_1 = 1$. Test $H_0: \beta_1 = 1$ against $H_1: \beta_1 \neq 1$ (the test with a given candidate value $b_1$ from Lecture 2) and also against the **one-sided alternative** (jednostranná alternativa) $H_1: \beta_1 < 1$: the lecture treats two-sided alternatives, for the one-sided one use the same statistic $T = (\widehat{\beta}_1 - 1)/\widehat{\sigma}(\widehat{\beta}_1)$, reject when $T < -t_{1-\alpha}(n-2)$, and the $p$-value is $P(t(n-2) < T)$. Check the duality of the two-sided test with the 95 % confidence interval from Task 02. Explain in your own words why Galton spoke of *regression* (regrese k průměru) and what the estimated slope says about sons of very tall and very short fathers.

95% interval spolehlivosti z Úlohy 2 neobsahuje hodnotu 1. Tzn mame zamítnout  $H_0$ na hladině významnosti $\alpha = 0.05$.

Víme, že $\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$. Pokud tento výraz dosadíme do základní rovnice pro predikci výšky syna ($\hat{y} = \hat{\beta}_0 + \hat{\beta}_1 x$), dostaneme $\hat{y} - \bar{y} = \hat{\beta}_1 (x - \bar{x})$. Pokud je otec výrazně vyšší než průměr, jeho odchylka $(x - \bar{x})$ je kladné číslo. Odchylka syna $(\hat{y} - \bar{y})$ je taky kladná, ale představuje pouze polovinu (51 %) odchylky otce.Pokud je otec výrazně nižší než průměr, jeho odchylka $(x - \bar{x})$ je záporné číslo. Když ji vynásobíme hodnotou $0.51$, získáme odchylku syna, jejíž absolutní hodnota se zmenší na polovinu. Tzn výška směřuje průměrné hodnotě z obou stran.

In [24]:

b1_candidate = 1.0
T = (beta1_hat - b1_candidate) / se_beta1
p_value_two_sided = 2 * stats.t.sf(abs(T), df=n - 2)

t_crit_one_sided = -stats.t.ppf(0.95, df=n - 2)
p_value_one_sided = stats.t.cdf(T, df=n - 2)

print(f"T-statistic: {T:.4f}")
print("\n--- Two-sided test ---")
print(f"p-value: {p_value_two_sided:.2e}")
print("Decision: Reject H0" if p_value_two_sided < 0.05 else "Decision: Do not reject H0")

print("\n--- One-sided test ---")
print(f"Critical value: {t_crit_one_sided:.4f}")
print(f"p-value: {p_value_one_sided:.2e}")
print("Decision: Reject H0" if T < t_crit_one_sided else "Decision: Do not reject H0")

T-statistic: -17.9641

--- Two-sided test ---
p-value: 2.61e-63
Decision: Reject H0

--- One-sided test ---
Critical value: -1.6463
p-value: 1.30e-63
Decision: Reject H0


## Task 05 - The intercept
Test $H_0: \beta_0 = 0$. Then write $\widehat{\beta}_0 = \overline{y} - \widehat{\beta}_1 \overline{x}$, plug in the sample means and explain why the intercept is far from zero even though sons are on average only about one inch taller than their fathers. Would a model through the origin make sense here?

Regrese k průměru snižuje směrnici téměř na polovinu. Kvůli tomu se od průměrné výšky synů neodečítá celá výška otců, ale pouze její polovina.Pokud donutíme přímku procházet bodem $(0,0)$, zcela to zkreslí směrnici v oblasti, kde se data skutečně nacházejí, a model přestane mít efekt regrese k průměru.

In [25]:

T0 = beta0_hat / se_beta0
p_value0 = 2 * stats.t.sf(abs(T0), df=n - 2)

print("--- Test H0: beta0 = 0 ---")
print(f"T-statistic: {T0:.4f}")
print(f"p-value: {p_value0:.2e}")
if p_value0 < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Do not reject H0")
x_bar = np.mean(fsdata['father'])
y_bar = np.mean(fsdata['son'])

print("\n--- Sample Means ---")
print(f"Mean of fathers (x_bar): {x_bar:.2f} inches")
print(f"Mean of sons (y_bar): {y_bar:.2f} inches")
print(f"Difference (y_bar - x_bar): {y_bar - x_bar:.2f} inches")

calculated_beta0 = y_bar - beta1_hat * x_bar
print("\n--- Plugging into the formula ---")
print(f"beta0 = y_bar - beta1_hat * x_bar")
print(f"beta0 = {y_bar:.2f} - ({beta1_hat:.4f} * {x_bar:.2f})")
print(f"beta0 = {calculated_beta0:.4f}")


--- Test H0: beta0 = 0 ---
T-statistic: 18.4935
p-value: 1.60e-66
Decision: Reject H0

--- Sample Means ---
Mean of fathers (x_bar): 67.69 inches
Mean of sons (y_bar): 68.68 inches
Difference (y_bar - x_bar): 1.00 inches

--- Plugging into the formula ---
beta0 = y_bar - beta1_hat * x_bar
beta0 = 68.68 - (0.5141 * 67.69)
beta0 = 33.8866


## Task 06 - Point predictions
Predict the mean height of sons whose fathers are 60, 64, 68, 72 and 76 inches tall and tabulate the difference *predicted son minus father*. For which father's height do the two coincide? (Confidence and prediction intervals for these predictions come in Lecture 4; here point predictions are enough.)

In [26]:
fathers_x = np.array([60, 64, 68, 72, 76])
sons_y_hat = beta0_hat + beta1_hat * fathers_x
diff = sons_y_hat - fathers_x

predictions_table = pd.DataFrame({
    "Father": fathers_x,
    "Predicted Son": sons_y_hat,
    "Difference (Son - Father)": diff
})

print("--- Predictions Table ---")
print(predictions_table.to_string(index=False))

coincide_point = beta0_hat / (1 - beta1_hat)

print("\n--- Point of Coincidence ---")
print(f"The heights coincide exactly at: {coincide_point:.2f} inches")

--- Predictions Table ---
 Father  Predicted Son  Difference (Son - Father)
     60      64.732187                   4.732187
     64      66.788559                   2.788559
     68      68.844931                   0.844931
     72      70.901303                  -1.098697
     76      72.957675                  -3.042325

--- Point of Coincidence ---
The heights coincide exactly at: 69.74 inches


## Task 07 - Reverse regression
Fit the reverse regression of `father` on `son` and report its slope. Verify numerically that the product of the two slopes equals the square of the sample correlation coefficient $r$ of the two heights (`np.corrcoef`), and explain why both slopes are smaller than 1. Is regression to the mean a causal effect of fathers on sons, or a statistical property of the two distributions?

Směrnice jsou menší než jedna jednoduše proto, že korelace je menší než jedna a rozptyly jsou stejné.Jde o statistickou vlastnost, protože tento efekt pozorujeme i v opačném směru.

In [27]:
S_yy = np.sum((y - y_bar) ** 2)
S_xy = np.sum((x - x_bar) * (y - y_bar))
slope_inverse_hat = S_xy / S_yy
print("--- Slopes ---")
print(f"Slope (son ~ father) [beta1_hat]: {beta1_hat:.4f}")
print(f"Slope (father ~ son) [slope_inverse_hat]: {slope_inverse_hat:.4f}")

r = np.corrcoef(x, y)[0, 1]
r_squared = r ** 2
product_of_slopes = beta1_hat * slope_inverse_hat
print("\n--- Numerical Verification ---")
print(f"Correlation coefficient (r): {r:.4f}")
print(f"r squared (r^2): {r_squared:.4f}")
print(f"Product of slopes: {product_of_slopes:.4f}")
print(f"Difference: {abs(r_squared - product_of_slopes):.2e}")


--- Slopes ---
Slope (son ~ father) [beta1_hat]: 0.5141
Slope (father ~ son) [slope_inverse_hat]: 0.4889

--- Numerical Verification ---
Correlation coefficient (r): 0.5013
r squared (r^2): 0.2513
Product of slopes: 0.2513
Difference: 2.04e-14


## Task 08 - Sample size and precision
Draw random subsamples of 30, 100 and 300 father-son pairs (fix the random seed), fit the model in each and compare the widths of the 95 % intervals for $\beta_1$ with the full-sample interval. Explain the pattern through $\mathrm{Var}(\widehat{\beta}_1) = \sigma^2 / S_{xx}$. *Optional:* repeat the subsampling with $n = 100$ five hundred times, treat the full-sample $\widehat{\beta}_1$ as the true value and estimate how often the 95 % interval covers it.

čím více lidí $n$ máme, tím více sčítanců v tomto součtu $S_{xx}$ je. Znamená to, že s rostoucí velikostí vzorku se jmenovatel $S_{xx}$ zvětšuje.A protože se jmenovatel zvětšuje, samotný zlomek (rozptyl $Var(\hat{\beta}_1)$) se zmenšuje. Menší rozptyl znamená menší chybu, což zužuje náš 95% interval spolehlivosti.

In [28]:

np.random.seed(57)

def fit(x_sample, y_sample):
    n_sample = len(x_sample)
    x_bar = np.mean(x_sample)
    y_bar = np.mean(y_sample)
    S_xx = np.sum((x_sample - x_bar)**2)
    S_xy = np.sum((x_sample - x_bar) * (y_sample - y_bar))
    beta1 = S_xy / S_xx
    beta0 = y_bar - beta1 * x_bar
    y_hat = beta0 + beta1 * x_sample
    residuals = y_sample - y_hat
    sigma2 = np.sum(residuals**2) / (n_sample - 2)

    se_beta1 = np.sqrt(sigma2 / S_xx)
    t_crit = stats.t.ppf(0.975, df=n_sample - 2)

    ci_lower = beta1 - t_crit * se_beta1
    ci_upper = beta1 + t_crit * se_beta1
    width = ci_upper - ci_lower
    return beta1, ci_lower, ci_upper, width

beta1_full, ci_low_full, ci_up_full, width_full = fit(x, y)
print("--- Confidence Interval Widths ---")
print(f"Full sample (n={len(x)}): width = {width_full:.4f}  CI = [{ci_low_full:.4f}, {ci_up_full:.4f}]")

sample_sizes = [30, 100, 300]
indices = np.arange(len(x))

for n in sample_sizes:
    idx = np.random.choice(indices, size=n, replace=False)
    _, ci_low, ci_up, width = fit(x[idx], y[idx])
    print(f"Subsample (n={n:<3}): width = {width:.4f}  CI = [{ci_low:.4f}, {ci_up:.4f}]")



--- Confidence Interval Widths ---
Full sample (n=1078): width = 0.1061  CI = [0.4610, 0.5672]
Subsample (n=30 ): width = 0.6067  CI = [0.3350, 0.9417]
Subsample (n=100): width = 0.3013  CI = [0.3751, 0.6763]
Subsample (n=300): width = 0.2157  CI = [0.4794, 0.6951]
